# 神经网络与深度学习：案例与实践 · 练习册

配套《神经网络与深度学习：案例与实践 —— 学习手册》，章节顺序对齐 [nndl/nndl-practice](https://github.com/nndl/nndl-practice) 的十章实践。

- **33 个练习**：每章 3–5 题，覆盖该章最关键的实现与原理；
- **10 道自测题**：每章一题；
- 每题带 `check()` 自检，失败会说明是哪一项；
- 全部题目只用 PyTorch，CPU 可运行，**不需要下载数据集**；
- 参考答案在最后一节（可执行代码）。

练习册不是仓库 notebook 的复制，而是把每章的核心组件拆成可独立验证的小题。


## 怎么用

1. 先读教材对应章节，再做本章练习；
2. 自己补全 `TODO`，再运行自测；
3. `[FAIL]` 先看提示，再对照错误信息；
4. 每章末尾运行 `scoreboard()` 看累计通过数；
5. 全部做完后再看参考答案。

## 章节对照

| 练习册 | 教材 | 仓库目录 |
| --- | --- | --- |
| 第 1 章 | 实践基础 | `pytorch/chap1实践基础/` |
| 第 2 章 | 机器学习概述 | `pytorch/chap2机器学习概述/` |
| 第 3 章 | 线性模型 | `pytorch/chap3线性模型/` |
| 第 4 章 | 前馈神经网络 | `pytorch/chap4前馈神经网络/` |
| 第 5 章 | 卷积神经网络 | `pytorch/chap5卷积神经网络/` |
| 第 6 章 | 循环神经网络 | `pytorch/chap6循环神经网络/` |
| 第 7 章 | 网络优化与正则化 | `pytorch/chap7网络优化与正则化/` |
| 第 8 章 | 注意力机制 | `pytorch/chap8注意力机制/` |
| 第 9 章 | 图神经网络 | `pytorch/chap9图神经网络/` |
| 第 10 章 | 大语言模型与智能体 | `pytorch/chap10大语言模型与智能体/` |


In [ ]:
# 环境与自测框架（直接运行）
import copy
import io
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)
random.seed(0)

_results = []


def check(name, fn):
    # fn 返回 True 通过；抛异常或返回 False 失败
    try:
        ok = bool(fn())
        err = None
    except Exception as e:
        ok = False
        err = e
    _results.append((name, ok))
    if ok:
        print(f"[PASS] {name}")
    else:
        detail = f"  <- {type(err).__name__}: {err}" if err else "  <- 返回 False"
        print(f"[FAIL] {name}{detail}")


def close(a, b, tol=1e-5):
    return torch.allclose(
        torch.as_tensor(a).float(), torch.as_tensor(b).float(), atol=tol, rtol=tol
    )


def quiz(name, given, answer):
    check(name, lambda: str(given).strip().upper() == answer.upper())


def n_params(m):
    return sum(p.numel() for p in m.parameters())


def scoreboard():
    passed = sum(ok for _, ok in _results)
    print(f"\n通过 {passed}/{len(_results)}")


print("PyTorch", torch.__version__, "|", "CUDA" if torch.cuda.is_available() else "CPU")


## 第 1 章　实践基础

### 练习 1.1　广播规则与报错

**要做什么**

1. 计算 `a + b`，存入 `out`；
2. 计算 `(2,3)` 与 `(2,1)` 相加的结果形状，存入 `shape_23_21`；
3. 用 `try/except` 捕获 `a + torch.ones(2)` 的异常类型名，存入 `err_type`。

**提示**：广播从尾部维度对齐，每个维度要么相等、要么为 1。


In [ ]:
a = torch.arange(6., dtype=torch.float32).reshape(2, 3)
b = torch.tensor([10., 20., 30.])

out = None            # TODO: a + b
shape_23_21 = None    # TODO: (2,3) + (2,1) 的结果形状

err_type = None
try:
    a + torch.ones(2)   # TODO: 保持这行，补全 except
except Exception as e:
    err_type = None   # TODO: type(e).__name__


In [ ]:
check("1.1a 广播结果正确", lambda: close(out, a + b))
check("1.1b (2,3)+(2,1) -> (2,3)", lambda: tuple(shape_23_21) == (2, 3))
check("1.1c 不兼容广播抛 RuntimeError", lambda: err_type == "RuntimeError")


### 练习 1.2　自动微分与链式法则

**要做什么**

1. 对 `x1` 计算 `(x1**3).sum()`，反向传播后把梯度存入 `grad1`；
2. 对 `w`、`b` 计算 `((w*x2 + b)**2).sum()`，反向传播后把梯度存入 `grad_w`、`grad_b`。

**提示**：标量输出可以直接 `backward()`；梯度会累加，注意只反传一次。


In [ ]:
x1 = torch.tensor([1., 2., 3.], requires_grad=True)
y1 = None       # TODO: (x1 ** 3).sum()
grad1 = None    # TODO: 反向传播后的 x1.grad

x2 = torch.tensor([1., 2., 3.], requires_grad=True)
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
loss2 = None    # TODO: ((w * x2 + b) ** 2).sum()
grad_w = None
grad_b = None


In [ ]:
check("1.2a x^3 的梯度为 3x^2",
      lambda: close(grad1, torch.tensor([3., 12., 27.])))
check("1.2b 复合函数对 w 的梯度 = 68",
      lambda: abs(grad_w.item() - 68.0) < 1e-5)
check("1.2c 复合函数对 b 的梯度 = 30",
      lambda: abs(grad_b.item() - 30.0) < 1e-5)


### 练习 1.3　用 `nn.Module` 实现 Linear

**要做什么**

实现 `MyLinear(in_f, out_f)`：

1. `__init__` 里创建可学习的 `weight`（形状 `(out_f, in_f)`）与 `bias`（形状 `(out_f,)`）；
2. `forward(x)` 返回 `x @ weight.T + bias`。

**提示**：参数要用 `nn.Parameter` 包装，才能被 `parameters()` 收集。


In [ ]:
class MyLinear(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        # TODO

    def forward(self, x):
        # TODO
        return None


linear = MyLinear(3, 4)
x_lin = torch.randn(5, 3)
out_lin = None   # TODO: linear(x_lin)


In [ ]:
check("1.3a 输出与 F.linear 一致",
      lambda: close(out_lin, F.linear(x_lin, linear.weight, linear.bias)))
check("1.3b 参数量 = in*out + out",
      lambda: n_params(linear) == 3 * 4 + 4)
check("1.3c 参数已注册（可被优化器看到）",
      lambda: len(list(linear.parameters())) == 2)


### 练习 1.4　`Dataset` 与 `DataLoader`

**要做什么**

补全 `ToyDataset`：

- `__len__` 返回样本数；
- `__getitem__(i)` 返回第 `i` 个 `(x, y)`；
- 然后用 `DataLoader(batch_size=4, shuffle=False)` 取出所有 batch，存入 `batches`（`list(loader)`）。

**提示**：数据是 `x = 0..9`，`y = 2x + 1`；10 个样本、batch=4，最后一批只有 2 个。


In [ ]:
class ToyDataset(Dataset):
    def __init__(self, n=10):
        self.x = torch.arange(n, dtype=torch.float32)
        self.y = 2 * self.x + 1

    def __len__(self):
        return None   # TODO

    def __getitem__(self, i):
        return None   # TODO


ds = ToyDataset()
loader = DataLoader(ds, batch_size=4, shuffle=False)
batches = None   # TODO: list(loader)


In [ ]:
check("1.4a 数据集长度为 10", lambda: len(ds) == 10)
check("1.4b batch 数为 3", lambda: len(loader) == 3)
check("1.4c 最后一个 batch 有 2 个样本",
      lambda: batches[-1][0].shape == (2,))
check("1.4d y = 2x + 1 在 batch 中保持",
      lambda: close(batches[0][1], 2 * batches[0][0] + 1))


In [ ]:
scoreboard()


**第 1 章要点**：广播看尾部维度；梯度会累加；`nn.Parameter` 才会被注册；`DataLoader` 的最后一个 batch 可能更小。


## 第 2 章　机器学习概述

### 练习 2.1　线性回归的闭式解

**要做什么**

给定 `X` 和 `y = 3X + 2`：

1. 构造设计矩阵 `A = [X, 1]`；
2. 用 `torch.linalg.lstsq(A, y).solution` 求参数 `w`（形状 `(2,1)`）；
3. 计算预测 `pred = A @ w`。

**提示**：`lstsq` 比直接求逆稳定；参数顺序是 `[斜率, 截距]`。


In [ ]:
X = torch.tensor([[1.], [2.], [3.], [4.]], dtype=torch.float32)
y = 3 * X + 2

A = None      # TODO: [X, 1]
w = None      # TODO: 最小二乘解
pred = None   # TODO: A @ w


In [ ]:
check("2.1a 预测拟合数据", lambda: close(pred, y, 1e-4))
check("2.1b 参数接近 [3, 2]",
      lambda: close(w.reshape(-1), torch.tensor([3., 2.]), 1e-3))


### 练习 2.2　多项式回归

**要做什么**

数据来自 $y = 1 - 2x + 3x^2$。构造特征矩阵 `[1, x, x^2]`，用最小二乘求系数 `coef`，并计算预测 `pred2`。

**提示**：`torch.stack([ones, x, x**2], dim=1)`，注意 `x` 的形状是 `(N,1)`。


In [ ]:
xs = torch.linspace(-2, 2, 9).unsqueeze(1)
ys = 1 - 2 * xs + 3 * xs ** 2

A2 = None      # TODO: [1, x, x^2]
coef = None    # TODO: 最小二乘解
pred2 = None   # TODO: A2 @ coef


In [ ]:
check("2.2a 预测拟合数据", lambda: close(pred2, ys, 1e-3))
check("2.2b 系数接近 [1, -2, 3]",
      lambda: close(coef.reshape(-1), torch.tensor([1., -2., 3.]), 1e-3))


### 练习 2.3　RunnerV1：把训练流程封装起来

**要做什么**

补全 `RunnerV1`，模仿仓库第 2 章的框架雏形：

1. `fit(X, y)`：用最小二乘解出 `model` 的权重与偏置；
2. `predict(X)`：前向推理；
3. `evaluate(X, y)`：返回 MSE；
4. `save(path)` / `load(path)`：用 `state_dict` 持久化（支持文件路径或 `BytesIO`）。

**提示**：模型是 `nn.Linear(1, 1)`；保存用 `torch.save(model.state_dict(), path)`，加载用 `load_state_dict(torch.load(path))`。下方的 `buf.seek(0)` 已经帮你处理了 `BytesIO` 的读指针。


In [ ]:
class RunnerV1:
    def __init__(self, model):
        self.model = model

    def fit(self, X, y):
        # TODO: 构造 [X, 1]，用 lstsq 求参数并写回 model
        return None

    def predict(self, X):
        return None

    def evaluate(self, X, y):
        return None

    def save(self, path):
        return None

    def load(self, path):
        return None


X_v1 = torch.tensor([[1.], [2.], [3.], [4.]], dtype=torch.float32)
y_v1 = 3 * X_v1 + 2

runner_v1 = RunnerV1(nn.Linear(1, 1))
runner_v1.fit(X_v1, y_v1)
pred_v1 = runner_v1.predict(X_v1)
mse_v1 = runner_v1.evaluate(X_v1, y_v1)

buf = io.BytesIO()
runner_v1.save(buf)
buf.seek(0)
runner_v1_loaded = RunnerV1(nn.Linear(1, 1))
runner_v1_loaded.load(buf)
pred_loaded = runner_v1_loaded.predict(X_v1)


In [ ]:
check("2.3a 拟合后 MSE 接近 0", lambda: mse_v1 < 1e-8)
check("2.3b 预测正确", lambda: close(pred_v1, y_v1, 1e-4))
check("2.3c 保存/加载后预测一致", lambda: close(pred_loaded, pred_v1, 1e-6))


In [ ]:
scoreboard()


**第 2 章要点**：闭式解适合小规模线性问题；多项式回归仍是线性模型；RunnerV1 的价值是把“训练—评价—预测—持久化”固定下来，后面只换模型和数据。


## 第 3 章　线性模型

### 练习 3.1　Logistic 回归的手写梯度

**要做什么**

实现 `logistic_grads(X, y, w, b)`，返回平均损失对 `w`、`b` 的梯度：

$$p=\sigma(Xw+b),\qquad \frac{\partial\mathcal{L}}{\partial z}=\frac{p-y}{N}$$

**提示**：`X` 形状 `(N,d)`，`y` 是 0/1 浮点；`grad_w = X.T @ dz`，`grad_b = dz.sum()`。不要调用 `backward()`。


In [ ]:
def logistic_grads(X, y, w, b):
    return None, None   # TODO


X3 = torch.tensor([[1., 2.], [2., -1.], [-1., 1.], [0., 3.]])
y3 = torch.tensor([1., 0., 0., 1.])
w3 = torch.tensor([0.5, -0.3])
b3 = torch.tensor(0.1)

grad_w = None   # TODO: logistic_grads(...)
grad_b = None


In [ ]:
# 用 autograd 计算参考梯度
_w = w3.clone().requires_grad_(True)
_b = b3.clone().requires_grad_(True)
_ref_loss = F.binary_cross_entropy_with_logits(X3 @ _w + _b, y3)
_ref_loss.backward()

check("3.1a grad_w 与 autograd 一致", lambda: close(grad_w, _w.grad, 1e-6))
check("3.1b grad_b 与 autograd 一致", lambda: close(grad_b, _b.grad, 1e-6))


### 练习 3.2　Softmax 回归的损失与梯度

**要做什么**

实现 `softmax_ce(logits, target)`，返回 `(loss, grad)`：

- `loss`：批次平均交叉熵；
- `grad`：损失对 logits 的梯度，形状与 logits 相同。

**提示**：用 log-sum-exp 保证稳定；梯度是 `(p - onehot) / N`。


In [ ]:
def softmax_ce(logits, target):
    return None, None   # TODO


logits3 = torch.tensor([[2.0, 0.5, -1.0], [0.0, 1.0, 2.0], [1.0, 1.0, 1.0]])
target3 = torch.tensor([0, 2, 1])

loss3 = None   # TODO
grad3 = None


In [ ]:
check("3.2a loss 与 F.cross_entropy 一致",
      lambda: close(loss3, F.cross_entropy(logits3, target3), 1e-6))
check("3.2b 梯度形状正确", lambda: tuple(grad3.shape) == tuple(logits3.shape))

_z = logits3.clone().requires_grad_(True)
_ce = F.cross_entropy(_z, target3)
_ce.backward()
check("3.2c 梯度与 autograd 一致", lambda: close(grad3, _z.grad, 1e-6))


### 练习 3.3　RunnerV2：验证集与最优模型

**要做什么**

补全 `RunnerV2`：

1. `train_epoch(loader)`：训练一个 epoch，返回平均损失；
2. `evaluate(loader)`：推理模式下的平均损失；
3. `fit(train_loader, dev_loader, epochs)`：记录 `{"train_loss": [...], "dev_loss": [...]}`，按 dev loss 保存最优 `state_dict` 到 `self.best_state`，返回 history。

**提示**：训练前 `model.train()`，评估用 `torch.no_grad()` 和 `model.eval()`；保存最优时用 `copy.deepcopy(model.state_dict())`。


In [ ]:
class RunnerV2:
    def __init__(self, model, optimizer, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.best_state = None
        self.best_dev = float("inf")

    def train_epoch(self, loader):
        return None   # TODO

    def evaluate(self, loader):
        return None   # TODO

    def fit(self, train_loader, dev_loader, epochs=8):
        return None   # TODO


torch.manual_seed(0)
Xc = torch.randn(200, 2)
yc = (Xc[:, 0] * Xc[:, 1] > 0).long()

train_loader = DataLoader(
    torch.utils.data.TensorDataset(Xc[:150], yc[:150]), batch_size=32, shuffle=True
)
dev_loader = DataLoader(
    torch.utils.data.TensorDataset(Xc[150:], yc[150:]), batch_size=32
)

model_v2 = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
runner_v2 = RunnerV2(
    model_v2,
    torch.optim.Adam(model_v2.parameters(), lr=0.05),
    nn.CrossEntropyLoss(),
)
history_v2 = runner_v2.fit(train_loader, dev_loader, epochs=8)


In [ ]:
check("3.3a history 记录了两个损失",
      lambda: set(history_v2.keys()) == {"train_loss", "dev_loss"})
check("3.3b 训练了 8 个 epoch", lambda: len(history_v2["dev_loss"]) == 8)
check("3.3c best_dev 等于最小的 dev loss",
      lambda: close(runner_v2.best_dev, min(history_v2["dev_loss"]), 1e-6))
check("3.3d 保存了最优 state_dict",
      lambda: runner_v2.best_state is not None and "0.weight" in runner_v2.best_state)
check("3.3e 训练损失下降",
      lambda: history_v2["train_loss"][-1] < history_v2["train_loss"][0] * 0.5)
check("3.3f 模型学会了 XOR（准确率 > 0.9）",
      lambda: (model_v2(Xc).argmax(-1) == yc).float().mean().item() > 0.9)


In [ ]:
scoreboard()


**第 3 章要点**：分类 = 线性输出 + sigmoid/softmax + 交叉熵；梯度是 `p - y`；RunnerV2 把验证集监控和最优模型保存带进流程。


## 第 4 章　前馈神经网络

### 练习 4.1　激活函数与导数

**要做什么**

1. 实现 `relu_t`、`gelu_t`（用 `torch.erf`）、`silu_t`；
2. 实现 `deriv(fn, x)`：用 autograd 返回 `fn` 在 `x` 处的导数。

**提示**：GELU 为 $x\Phi(x)=\frac{x}{2}[1+\mathrm{erf}(x/\sqrt2)]$；SiLU 为 $x\sigma(x)$。


In [ ]:
def relu_t(x):
    return None   # TODO

def gelu_t(x):
    return None   # TODO

def silu_t(x):
    return None   # TODO

def deriv(fn, x):
    xg = x.detach().clone().requires_grad_(True)
    y = fn(xg)
    return None   # TODO: torch.autograd.grad(...)


In [ ]:
t4 = torch.linspace(-4, 4, 101)

check("4.1a GELU 与 F.gelu 一致", lambda: close(gelu_t(t4), F.gelu(t4), 1e-6))
check("4.1b GELU 在 0 处导数为 0.5",
      lambda: abs(deriv(gelu_t, torch.tensor(0.0)).item() - 0.5) < 1e-6)
check("4.1c SiLU 在 0 处导数为 0.5",
      lambda: abs(deriv(silu_t, torch.tensor(0.0)).item() - 0.5) < 1e-6)
check("4.1d ReLU 在 -1 处导数为 0",
      lambda: abs(deriv(relu_t, torch.tensor(-1.0)).item()) < 1e-8)


### 练习 4.2　手算两层 MLP 的反向传播

**要做什么**

网络：`z1 = XW1 + b1`，`a1 = ReLU(z1)`，`z2 = a1W2 + b2`，MSE 损失。

不用 autograd，手算 `loss4` 以及 `dW1、db1、dW2、db2`。

**提示**：

- $\partial\mathcal{L}/\partial z_2=\frac{2}{N}(z_2-y)$
- $\partial\mathcal{L}/\partial W_2=a_1^{\top}\partial z_2$，$\partial\mathcal{L}/\partial b_2=\sum\partial z_2$
- $\partial\mathcal{L}/\partial z_1=(\partial z_2W_2^{\top})\odot\mathbb{1}[z_1>0]$
- $\partial\mathcal{L}/\partial W_1=X^{\top}\partial z_1$，$\partial\mathcal{L}/\partial b_1=\sum\partial z_1$


In [ ]:
torch.manual_seed(0)
X4 = torch.randn(6, 2)
y4 = torch.randn(6, 1)
W1_4 = torch.randn(2, 3)
b1_4 = torch.zeros(3)
W2_4 = torch.randn(3, 1)
b2_4 = torch.zeros(1)

z1 = None    # TODO
a1 = None    # TODO
z2 = None    # TODO
loss4 = None # TODO

dW1 = None    # TODO
db1 = None
dW2 = None
db2 = None


In [ ]:
# autograd 参考
_X = X4.clone().requires_grad_(False)
_W1 = W1_4.clone().requires_grad_(True)
_b1 = b1_4.clone().requires_grad_(True)
_W2 = W2_4.clone().requires_grad_(True)
_b2 = b2_4.clone().requires_grad_(True)
_ref_z1 = _X @ _W1 + _b1
_ref_a1 = torch.relu(_ref_z1)
_ref_z2 = _ref_a1 @ _W2 + _b2
_ref_loss = F.mse_loss(_ref_z2, y4)
_ref_loss.backward()

check("4.2a 前向 z1 正确", lambda: close(z1, _ref_z1))
check("4.2b 前向 z2 正确", lambda: close(z2, _ref_z2))
check("4.2c loss 正确", lambda: abs(loss4.item() - _ref_loss.item()) < 1e-6)
check("4.2d dW2 正确", lambda: close(dW2, _W2.grad, 1e-6))
check("4.2e db2 正确", lambda: close(db2.reshape(-1), _b2.grad.reshape(-1), 1e-6))
check("4.2f dW1 正确", lambda: close(dW1, _W1.grad, 1e-6))
check("4.2g db1 正确", lambda: close(db1.reshape(-1), _b1.grad.reshape(-1), 1e-6))


### 练习 4.3　RunnerV3 的四个零件

**要做什么**

实现仓库 RunnerV3 依赖的四个小函数：

1. `accuracy(logits, y)`：返回准确率（Python float）；
2. `is_better(cur, best, higher_is_better)`：判断当前指标是否更优；
3. `train_epoch_v3(model, loader, optimizer, loss_fn)`：训练一个 epoch，返回平均损失；
4. `evaluate_v3(model, loader, loss_fn, metric_fn)`：返回 `(平均损失, 指标)`。

**提示**：`higher_is_better=False` 时越小越好；评估用 `no_grad` + `eval()`。


In [ ]:
def accuracy(logits, y):
    return None   # TODO

def is_better(cur, best, higher_is_better):
    return None   # TODO

def train_epoch_v3(model, loader, optimizer, loss_fn):
    return None   # TODO

def evaluate_v3(model, loader, loss_fn, metric_fn):
    return None   # TODO


torch.manual_seed(0)
X4b = torch.randn(200, 2)
y4b = (X4b[:, 0] * X4b[:, 1] > 0).long()
loader4 = DataLoader(torch.utils.data.TensorDataset(X4b, y4b), batch_size=32, shuffle=True)
model4 = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
opt4 = torch.optim.Adam(model4.parameters(), lr=0.05)
loss4b = nn.CrossEntropyLoss()

first_loss = train_epoch_v3(model4, loader4, opt4, loss4b)
for _ in range(2):
    train_epoch_v3(model4, loader4, opt4, loss4b)
try:
    dev_loss4, dev_acc4 = evaluate_v3(model4, loader4, loss4b, accuracy)
except Exception:
    dev_loss4, dev_acc4 = None, None


In [ ]:
check("4.3a accuracy 计算正确",
      lambda: abs(accuracy(torch.tensor([[2., 0.], [0., 2.], [2., 0.], [0., 2.]]),
                           torch.tensor([0, 1, 1, 1])) - 0.75) < 1e-9)
check("4.3b is_better：指标越高越好",
      lambda: is_better(0.9, 0.8, True) and not is_better(0.7, 0.8, True))
check("4.3c is_better：损失越低越好",
      lambda: is_better(0.5, 0.6, False) and not is_better(0.7, 0.6, False))
check("4.3d evaluate 返回 (loss, metric)",
      lambda: isinstance(dev_loss4, float) and isinstance(dev_acc4, float))
check("4.3e 训练一个 epoch 后 loss 为有限值",
      lambda: math.isfinite(first_loss))
check("4.3f 模型学得比随机好（准确率 > 0.8）",
      lambda: dev_acc4 > 0.8)


In [ ]:
scoreboard()


**第 4 章要点**：激活函数提供非线性；反向传播就是链式法则；RunnerV3 把 loss 与 metric 分开，用 metric 选最优模型。


## 第 5 章　卷积神经网络

### 练习 5.1　用 unfold 从零实现 conv2d

**要做什么**

实现 `conv2d_naive(x, weight, bias=None, stride=1, padding=0)`：

1. 用 `F.unfold` 把输入按滑动窗口展开；
2. 把权重 reshape 成 `(out_c, C*k*k)` 后做矩阵乘；
3. 变回 `(N, out_c, H_out, W_out)`，再加上偏置。

**提示**：`F.unfold(x, kernel_size=k, padding=p, stride=s)` 返回 `(N, C*k*k, L)`；`L` 是输出位置的个数，`H_out = (H + 2p - k)//s + 1`。


In [ ]:
def conv2d_naive(x, weight, bias=None, stride=1, padding=0):
    return None   # TODO


torch.manual_seed(0)
x5 = torch.randn(2, 3, 8, 8)
w5 = torch.randn(4, 3, 3, 3)
b5 = torch.randn(4)

out_same = conv2d_naive(x5, w5, b5, stride=1, padding=1)
out_down = conv2d_naive(x5, w5, b5, stride=2, padding=0)


In [ ]:
check("5.1a stride=1 padding=1 与 F.conv2d 一致",
      lambda: close(out_same, F.conv2d(x5, w5, b5, stride=1, padding=1), 1e-4))
check("5.1b stride=2 padding=0 与 F.conv2d 一致",
      lambda: close(out_down, F.conv2d(x5, w5, b5, stride=2, padding=0), 1e-4))


### 练习 5.2　LeNet-5 的形状与参数量

**要做什么**

输入 `1×32×32`，补全每一层的 `(输出形状, 参数量)`：

| 层 | 操作 |
| --- | --- |
| C1 | 6 个 5×5 卷积 |
| S2 | 2×2 最大池化 |
| C3 | 16 个 5×5 卷积 |
| S4 | 2×2 平均池化 |
| C5 | 120 个 5×5 卷积 |
| F6 | 120→84 全连接 |
| F7 | 84→10 全连接 |

**提示**：卷积参数量为 `out_c * (in_c*k*k + 1)`；池化没有参数；总参数量应约为 6.2 万。


In [ ]:
lenet_shapes = {
    "C1": None,   # TODO: ((6,28,28), 参数量)
    "S2": None,
    "C3": None,
    "S4": None,
    "C5": None,
    "F6": None,
    "F7": None,
}
lenet_total = None   # TODO: 总参数量


In [ ]:
check("5.2a C1 形状与参数", lambda: lenet_shapes["C1"] == ((6, 28, 28), 156))
check("5.2b S2 形状与参数", lambda: lenet_shapes["S2"] == ((6, 14, 14), 0))
check("5.2c C3 形状与参数", lambda: lenet_shapes["C3"] == ((16, 10, 10), 2416))
check("5.2d S4 形状与参数", lambda: lenet_shapes["S4"] == ((16, 5, 5), 0))
check("5.2e C5 形状与参数", lambda: lenet_shapes["C5"] == ((120, 1, 1), 48120))
check("5.2f F6 形状与参数", lambda: lenet_shapes["F6"] == ((84,), 10164))
check("5.2g F7 形状与参数", lambda: lenet_shapes["F7"] == ((10,), 850))
check("5.2h 总参数量 61706", lambda: lenet_total == 61706)


### 练习 5.3　残差连接的梯度效应

**要做什么**

实现 `grad_norm_block(depth, use_residual, seed=0)`：

1. 构建 `depth` 个 `nn.Linear(64, 64)`，全部用 Xavier 初始化；
2. 逐层 `h = act(linear(h))`；若 `use_residual=True`，改为 `h = h + act(linear(h))`；
3. 对 `h.pow(2).mean()` 反向传播，返回输入梯度的范数。

**提示**：这就是仓库第 5 章“ResNet18 去掉残差后变差”的最小实验。


In [ ]:
def grad_norm_block(depth, use_residual, seed=0):
    return None   # TODO


g_nores_20 = None   # TODO: grad_norm_block(20, False)
g_res_20 = None     # TODO: grad_norm_block(20, True)


In [ ]:
check("5.3a 无残差时 20 层梯度极小（< 1e-4）",
      lambda: g_nores_20 < 1e-4)
check("5.3b 残差让梯度保持非零（> 1e-2）",
      lambda: g_res_20 > 1e-2)
check("5.3c 残差的梯度比无残差大至少 3 个数量级",
      lambda: g_res_20 > g_nores_20 * 1e3)


In [ ]:
scoreboard()


**第 5 章要点**：卷积输出形状与参数量要能手算；im2col 是工业实现的本质；残差解决退化，不是过拟合；LeNet 的参数主要在全连接层。


## 第 6 章　循环神经网络

### 练习 6.1　手写 SRN 并与 `nn.RNN` 对齐

**要做什么**

实现 `MySRN.forward(x)`：

$$h_t=\tanh(x_tW_{xh}^{\top}+b_{ih}+h_{t-1}W_{hh}^{\top}+b_{hh})$$

输入 `x` 形状 `(B,T,D)`，返回 `(outputs, h_last)`，其中 `outputs` 形状 `(B,T,H)`，`h_last` 形状 `(1,B,H)`。

**提示**：初始隐藏状态是全零；`nn.RNN` 使用 `batch_first=True` 时输出也是 `(B,T,H)`。


In [ ]:
class MySRN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_xh = nn.Parameter(torch.randn(hidden_size, input_size))
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.b_ih = nn.Parameter(torch.zeros(hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x):
        # x: (B, T, D) -> outputs (B, T, H), h_last (1, B, H)
        return None, None   # TODO


torch.manual_seed(0)
rnn6 = nn.RNN(3, 5, batch_first=True)
srn6 = MySRN(3, 5)
with torch.no_grad():
    srn6.W_xh.copy_(rnn6.weight_ih_l0)
    srn6.W_hh.copy_(rnn6.weight_hh_l0)
    srn6.b_ih.copy_(rnn6.bias_ih_l0)
    srn6.b_hh.copy_(rnn6.bias_hh_l0)

x6 = torch.randn(2, 4, 3)
try:
    out_srn, h_srn = srn6(x6)
except Exception:
    out_srn, h_srn = None, None


In [ ]:
out_ref6, h_ref6 = rnn6(x6)
check("6.1a 输出形状为 (2,4,5)",
      lambda: tuple(out_srn.shape) == (2, 4, 5))
check("6.1b 所有时间步输出与 nn.RNN 一致",
      lambda: close(out_srn, out_ref6, 1e-5))
check("6.1c 最后隐藏状态一致",
      lambda: close(h_srn.reshape(2, 5), h_ref6.reshape(2, 5), 1e-5))


### 练习 6.2　手写 LSTM 单步

**要做什么**

用 `nn.LSTM` 的权重手算 $t=0$ 的隐藏状态（初始 $h_0=c_0=0$）：

$$i,f,g,o=\mathrm{chunk}(x_tW_{ih}^{\top}+hW_{hh}^{\top}+b_{ih}+b_{hh},4)$$

$$c=f\odot c_0+i\odot\tanh(g),\qquad h=o\odot\tanh(c)$$

**提示**：PyTorch 的门顺序是 **i, f, g, o**。


In [ ]:
torch.manual_seed(0)
lstm6 = nn.LSTM(4, 5)
x6b = torch.randn(1, 1, 4)
lstm_out6, _ = lstm6(x6b)

h_manual6 = None   # TODO


In [ ]:
check("6.2 手算 h 与 nn.LSTM 一致", lambda: close(lstm_out6[0], h_manual6, 1e-5))


### 练习 6.3　梯度裁剪

**要做什么**

实现 `clip_grads(params, max_norm)`：把所有参数的梯度按总范数裁剪到 `max_norm`，**原地修改**，并返回裁剪前的总范数。

$$g\leftarrow g\cdot\min\left(1,\frac{\text{max\_norm}}{\lVert g\rVert}\right)$$

**提示**：总范数是对所有参数梯度拼接后的 L2 范数；`torch.nn.utils.clip_grad_norm_` 的语义相同，但这里自己实现。


In [ ]:
def clip_grads(params, max_norm):
    return None   # TODO


torch.manual_seed(0)
params6 = [torch.randn(4, requires_grad=True) for _ in range(3)]
for p in params6:
    p.grad = torch.randn(4)

grad_before = torch.cat([p.grad.flatten() for p in params6]).clone()
norm_before = clip_grads(params6, 1.0)
grad_after = torch.cat([p.grad.flatten() for p in params6])


In [ ]:
check("6.3a 返回裁剪前的总范数",
      lambda: abs(norm_before - grad_before.norm().item()) < 1e-5)
check("6.3b 裁剪后范数不超过 1",
      lambda: grad_after.norm().item() <= 1.0 + 1e-5)
check("6.3c 方向保持不变（余弦相似度接近 1）",
      lambda: abs(norm_before - grad_before.norm().item()) < 1e-5
              and grad_after.norm().item() <= 1.0 + 1e-5
              and torch.nn.functional.cosine_similarity(
                  grad_before.unsqueeze(0), grad_after.unsqueeze(0)).item() > 0.999)


In [ ]:
scoreboard()


**第 6 章要点**：SRN 的权重布局要与 `nn.RNN` 对齐；LSTM 的门顺序是 i,f,g,o；梯度裁剪只改模长不改方向。


## 第 7 章　网络优化与正则化

### 练习 7.1　实现一步 Adam

**要做什么**

实现 `adam_step(p, grad, m, v, t, lr=0.01, b1=0.9, b2=0.999, eps=1e-8)`，返回更新后的 `(p, m, v)`：

$$
\begin{aligned}
m&\leftarrow b_1m+(1-b_1)g\\
v&\leftarrow b_2v+(1-b_2)g^2\\
\hat m&=\frac{m}{1-b_1^t},\qquad \hat v=\frac{v}{1-b_2^t}\\
p&\leftarrow p-\eta\frac{\hat m}{\sqrt{\hat v}+\epsilon}
\end{aligned}
$$

**提示**：`t` 从 1 开始；返回的是新张量，不要原地改动输入（便于比较）。


In [ ]:
def adam_step(p, grad, m, v, t, lr=0.01, b1=0.9, b2=0.999, eps=1e-8):
    return None, None, None   # TODO


torch.manual_seed(0)
p0 = torch.randn(3)
g0 = torch.randn(3)
m0 = torch.zeros(3)
v0 = torch.zeros(3)

try:
    p_new, m_new, v_new = adam_step(p0.clone(), g0, m0, v0, t=1, lr=0.01)
except Exception:
    p_new, m_new, v_new = None, None, None


In [ ]:
p_ref = nn.Parameter(p0.clone())
opt_ref = torch.optim.Adam([p_ref], lr=0.01)
p_ref.grad = g0.clone()
opt_ref.step()
state = opt_ref.state[p_ref]

check("7.1a 参数更新与 Adam 一致", lambda: close(p_new, p_ref.detach(), 1e-6))
check("7.1b 一阶矩一致", lambda: close(m_new, state["exp_avg"], 1e-6))
check("7.1c 二阶矩一致", lambda: close(v_new, state["exp_avg_sq"], 1e-6))


### 练习 7.2　初始化与方差

**要做什么**

实现 `std_ratio(init_fn, depth=30, width=256)`：

- 构建 `depth` 层 `nn.Linear(width, width)`，每层用 `init_fn` 初始化权重；
- 激活用 ReLU；返回“最后一层输出标准差 / 第一层输出标准差”。

**提示**：He 初始化应接近 1；PyTorch 默认初始化会随层数快速衰减。


In [ ]:
def std_ratio(init_fn, depth=30, width=256):
    return None   # TODO


he_init = lambda lin: nn.init.kaiming_normal_(lin.weight, nonlinearity="relu")
default_init = lambda lin: nn.init.kaiming_uniform_(lin.weight, a=math.sqrt(5))

ratio_he = None       # TODO: std_ratio(he_init)
ratio_default = None  # TODO: std_ratio(default_init)


In [ ]:
check("7.2a He 初始化保持方差（> 0.5）", lambda: ratio_he > 0.5)
check("7.2b 默认初始化在深层 ReLU 栈中衰减（< 0.2）",
      lambda: ratio_default < 0.2)


### 练习 7.3　手写 BatchNorm 前向

**要做什么**

实现 `bn_forward(x, gamma, beta, running_mean, running_var, momentum=0.1, eps=1e-5, training=True)`，返回 `(y, new_running_mean, new_running_var)`：

- 训练模式：用当前 batch 的均值/总体方差归一化，并按动量更新 running 统计量；
- 推理模式：用传入的 running 统计量归一化，不更新。

**提示**：归一化时用的是总体方差（`unbiased=False`），但 PyTorch 更新 `running_var` 时用的是无偏方差（`unbiased=True`）——这是实现 BatchNorm 时最容易踩的坑。更新公式为 `new = (1-momentum) * old + momentum * batch_stat`。


In [ ]:
def bn_forward(x, gamma, beta, running_mean, running_var,
               momentum=0.1, eps=1e-5, training=True):
    return None, None, None   # TODO


torch.manual_seed(0)
bn_ref = nn.BatchNorm1d(4)
x_bn = torch.randn(8, 4) * 2 + 5
gamma, beta = bn_ref.weight.detach().clone(), bn_ref.bias.detach().clone()

bn_ref.train()
y_train_ref = bn_ref(x_bn)
rm_ref = bn_ref.running_mean.clone()
rv_ref = bn_ref.running_var.clone()

try:
    y_train_mine, rm_mine, rv_mine = bn_forward(
        x_bn, gamma, beta, torch.zeros(4), torch.ones(4), training=True
    )
except Exception:
    y_train_mine = rm_mine = rv_mine = None

bn_ref.eval()
y_eval_ref = bn_ref(x_bn)
try:
    y_eval_mine, _, _ = bn_forward(
        x_bn, gamma, beta, rm_ref, rv_ref, training=False
    )
except Exception:
    y_eval_mine = None


In [ ]:
check("7.3a 训练模式输出与 nn.BatchNorm1d 一致",
      lambda: close(y_train_mine, y_train_ref, 1e-5))
check("7.3b running_mean 更新正确",
      lambda: close(rm_mine, rm_ref, 1e-5))
check("7.3c running_var 更新正确",
      lambda: close(rv_mine, rv_ref, 1e-5))
check("7.3d 推理模式使用 running 统计量",
      lambda: close(y_eval_mine, y_eval_ref, 1e-5))


In [ ]:
scoreboard()


**第 7 章要点**：Adam 的偏差修正不要漏；初始化要匹配激活；BN 训练用 batch 统计、推理用滑动统计，且方差是总体方差。


## 第 8 章　注意力机制

### 练习 8.1　加性注意力与缩放点积注意力

**要做什么**

实现：

1. `additive_attention(q, K, V, Wq, Wk, v)`：$e_i=v^{\top}\tanh(W_q q+W_k k_i)$，softmax 后加权求和；
2. `scaled_dot_attention(q, K, V)`：$e_i=q^{\top}k_i/\sqrt{d}$。

**提示**：`K @ Wk.T` 一次得到所有键的投影；softmax 沿最后一维。


In [ ]:
def additive_attention(q, K, V, Wq, Wk, v):
    return None   # TODO

def scaled_dot_attention(q, K, V):
    return None   # TODO


torch.manual_seed(0)
q8 = torch.randn(4)
K8 = torch.randn(5, 4)
V8 = torch.randn(5, 2)
Wq8 = torch.randn(3, 4)
Wk8 = torch.randn(3, 4)
v8 = torch.randn(3)

ctx_add = None   # TODO
ctx_dot = None   # TODO


In [ ]:
# 参考实现
_scores_add = torch.tanh(Wq8 @ q8 + K8 @ Wk8.T) @ v8
_ctx_add = _scores_add.softmax(-1) @ V8
_scores_dot = (K8 @ q8) / math.sqrt(q8.size(0))
_ctx_dot = _scores_dot.softmax(-1) @ V8

check("8.1a 加性注意力结果正确", lambda: close(ctx_add, _ctx_add, 1e-6))
check("8.1b 缩放点积注意力结果正确", lambda: close(ctx_dot, _ctx_dot, 1e-6))
check("8.1c 输出形状为 (2,)", lambda: tuple(ctx_add.shape) == (2,))


### 练习 8.2　多头自注意力

**要做什么**

实现 `MultiHeadAttention(d_model, n_heads)`：

1. `__init__` 定义 `q_proj`、`k_proj`、`v_proj`、`out_proj` 四个 `nn.Linear(d_model, d_model)`；
2. `forward(x, causal=False)`：拆头 → 缩放点积注意力 → 合并 → 输出投影；
3. `causal=True` 时加因果掩码。

**提示**：拆头用 `view(B, T, H, dh).transpose(1, 2)`；合并用 `transpose(1, 2).reshape(B, T, d)`。


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        # TODO: 四个线性层

    def forward(self, x, causal=False):
        # TODO
        return None


torch.manual_seed(0)
mha8 = MultiHeadAttention(12, 3)
x8 = torch.randn(2, 5, 12)
try:
    out8 = mha8(x8, causal=True)
    x8b = x8.clone()
    x8b[:, -1] += 1.0
    out8b = mha8(x8b, causal=True)
except Exception:
    out8, out8b = None, None


In [ ]:
check("8.2a 输出形状为 (2,5,12)", lambda: tuple(out8.shape) == (2, 5, 12))
check("8.2b 参数量 = 4*d^2 + 4*d",
      lambda: n_params(mha8) == 4 * 12 * 12 + 4 * 12)
check("8.2c 因果掩码：改变最后一个 token 不影响前面的输出",
      lambda: close(out8b[:, :-1], out8[:, :-1], 1e-6))


### 练习 8.3　正弦位置编码

**要做什么**

实现 `sinusoidal_pe(seq_len, d_model)`：

$$PE_{(pos,2i)}=\sin\left(\frac{pos}{10000^{2i/d}}\right),\qquad PE_{(pos,2i+1)}=\cos\left(\frac{pos}{10000^{2i/d}}\right)$$

**提示**：先算频率向量 `theta = 10000 ** (-2i/d)`，再与位置相乘；用 `torch.arange` 构造网格。


In [ ]:
def sinusoidal_pe(seq_len, d_model):
    return None   # TODO


pe8 = sinusoidal_pe(6, 8)


In [ ]:
check("8.3a 形状为 (6,8)", lambda: tuple(pe8.shape) == (6, 8))
check("8.3b 第 0 行的偶数列为 0、奇数列为 1",
      lambda: close(pe8[0], torch.tensor([0., 1., 0., 1., 0., 1., 0., 1.]), 1e-6))
check("8.3c 第 0 列为 sin(pos)、第 1 列为 cos(pos)",
      lambda: close(pe8[:, 0], torch.sin(torch.arange(6.).float()), 1e-6)
              and close(pe8[:, 1], torch.cos(torch.arange(6.).float()), 1e-6))
check("8.3d 第 2 列的频率为 10000^(-2/d)",
      lambda: abs(pe8[1, 2].item() - math.sin(10000 ** (-0.25))) < 1e-6)


In [ ]:
scoreboard()


**第 8 章要点**：注意力 = 打分 + softmax + 加权；缩放因子控制 softmax 的饱和；多头在多个子空间并行；因果掩码保证自回归；位置编码补齐顺序信息。


## 第 9 章　图神经网络

### 练习 9.1　GCN 层

**要做什么**

实现 `gcn_layer(A, H, W)`，返回 `(out, A_norm)`：

$$\tilde A=A+I,\qquad A_{norm}=\tilde D^{-1/2}\tilde A\tilde D^{-1/2},\qquad out=\mathrm{ReLU}(A_{norm}HW)$$

**提示**：`A` 是 0/1 邻接矩阵（不含自环）；`D` 是 $\tilde A$ 的度矩阵。


In [ ]:
def gcn_layer(A, H, W):
    return None, None   # TODO


torch.manual_seed(0)
N9 = 6
A9 = torch.zeros(N9, N9)
for i in range(N9):
    A9[i, (i + 1) % N9] = 1.0
    A9[(i + 1) % N9, i] = 1.0
H9 = torch.randn(N9, 4)
W9 = torch.randn(4, 5)

gcn_out9 = None    # TODO
gcn_A_norm9 = None


In [ ]:
A_tilde9 = A9 + torch.eye(N9)
D9 = A_tilde9.sum(-1)
ref_A9 = torch.diag(D9.pow(-0.5)) @ A_tilde9 @ torch.diag(D9.pow(-0.5))
ref_out9 = F.relu(ref_A9 @ H9 @ W9)

check("9.1a 归一化邻接矩阵正确", lambda: close(gcn_A_norm9, ref_A9))
check("9.1b 归一化邻接矩阵对称", lambda: close(gcn_A_norm9, gcn_A_norm9.T))
check("9.1c GCN 输出正确", lambda: close(gcn_out9, ref_out9))


### 练习 9.2　图注意力层（GAT）

**要做什么**

实现 `gat_layer(H, A, W, a, negative_slope=0.2)`，返回 `(out, alpha)`：

$$e_{ij}=\mathrm{LeakyReLU}\left(a_{1}^{\top}Wh_i+a_{2}^{\top}Wh_j\right)$$

只保留 `A[i,j]=True` 的边，按行 softmax 得到 $\alpha$，再聚合并过 ReLU。

**提示**：`a` 长度是 `2F`；`e = term_i[:,None] + term_j[None,:]`；非边用 `masked_fill(-inf)`。


In [ ]:
def gat_layer(H, A, W, a, negative_slope=0.2):
    return None, None   # TODO


torch.manual_seed(0)
A9b = torch.tensor([
    [1, 1, 0, 0],
    [1, 1, 1, 0],
    [0, 1, 1, 1],
    [0, 0, 1, 1],
], dtype=torch.bool)
H9b = torch.randn(4, 3)
W9b = torch.randn(3, 2)
a9b = torch.randn(4)

gat_out9 = None    # TODO
gat_alpha9 = None


In [ ]:
_Wh = H9b @ W9b
_e = F.leaky_relu(_Wh @ a9b[:2].unsqueeze(1) + (_Wh @ a9b[2:].unsqueeze(1)).T, 0.2)
_e = _e.masked_fill(~A9b, float("-inf"))
_ref_alpha = _e.softmax(-1)
_ref_out = F.relu(_ref_alpha @ _Wh)

check("9.2a 注意力每行和为 1", lambda: close(gat_alpha9.sum(-1), torch.ones(4), 1e-5))
check("9.2b 非边位置的注意力为 0",
      lambda: gat_alpha9[~A9b].abs().max().item() < 1e-8)
check("9.2c GAT 输出正确", lambda: close(gat_out9, _ref_out, 1e-6))


### 练习 9.3　过平滑

**要做什么**

实现：

1. `mean_cosine(X)`：返回节点表示两两余弦相似度的平均值（不含对角线）；
2. `smooth_sim(A_norm, X, layers)`：把 `X` 反复乘以 `A_norm`，返回最终的 `mean_cosine`。

**提示**：`F.normalize(X, dim=-1)` 后做 `X @ X.T`；8 层的相似度应明显高于 2 层。


In [ ]:
def mean_cosine(X):
    return None   # TODO

def smooth_sim(A_norm, X, layers):
    return None   # TODO


N9c = 10
A9c = torch.zeros(N9c, N9c)
for i in range(N9c):
    A9c[i, i] = 1.0
    A9c[i, (i + 1) % N9c] = 1.0
    A9c[i, (i - 1) % N9c] = 1.0
D9c = A9c.sum(-1)
A_norm9c = torch.diag(D9c.pow(-0.5)) @ A9c @ torch.diag(D9c.pow(-0.5))
torch.manual_seed(0)
X9c = torch.randn(N9c, 8)

sim2 = None   # TODO: smooth_sim(A_norm9c, X9c, 2)
sim8 = None   # TODO: smooth_sim(A_norm9c, X9c, 8)


In [ ]:
check("9.3a 相似度在 [0,1] 内", lambda: 0 <= sim2 <= 1 and 0 <= sim8 <= 1)
check("9.3b 8 层比 2 层更平滑", lambda: sim8 > sim2 + 0.02)
check("9.3c 8 层后节点表示明显趋同", lambda: sim8 > 0.75)


In [ ]:
scoreboard()


**第 9 章要点**：GCN 的对称归一化邻接矩阵是核心；GAT 用注意力学邻居权重；层数过多会过平滑，2–4 层通常够用。


## 第 10 章　大语言模型与智能体

### 练习 10.1　因果自注意力

**要做什么**

实现 `causal_attention(q, k, v)`，输入形状 `(T, d)`，返回 `(out, weights)`：

$$\mathrm{weights}=\mathrm{softmax}\left(\frac{qk^{\top}}{\sqrt{d}}+\text{mask}\right),\qquad out=\mathrm{weights}\,v$$

掩码：位置 $i$ 只能看到 $j\le i$。

**提示**：`torch.triu(..., diagonal=1)` 是上三角 mask；masked_fill 用 `-inf`。


In [ ]:
def causal_attention(q, k, v):
    return None, None   # TODO


torch.manual_seed(0)
T10 = 6
q10 = torch.randn(T10, 8)
k10 = torch.randn(T10, 8)
v10 = torch.randn(T10, 8)

attn_out10 = None   # TODO
attn_w10 = None


In [ ]:
_mask10 = torch.triu(torch.ones(T10, T10, dtype=torch.bool), diagonal=1)
_ref_w10 = (q10 @ k10.T / math.sqrt(8)).masked_fill(_mask10, float("-inf")).softmax(-1)

check("10.1a 未来位置权重为 0", lambda: attn_w10[_mask10].abs().max().item() < 1e-8)
check("10.1b 每行权重和为 1", lambda: close(attn_w10.sum(-1), torch.ones(T10), 1e-5))
check("10.1c 权重与参考实现一致", lambda: close(attn_w10, _ref_w10, 1e-6))
check("10.1d 输出等于 weights @ v", lambda: close(attn_out10, attn_w10 @ v10, 1e-6))


### 练习 10.2　KV Cache

**要做什么**

实现 `KVCacheAttention.forward(x, cache=None)`：

- `x` 是**新 token**（形状 `(T_new, d)`）；
- `cache` 是 `(K_prev, V_prev)` 或 `None`；
- 计算新 token 的 Q/K/V，把 K/V 拼接到 cache 后，对全部 K/V 做注意力；
- 返回 `(out, (K_all, V_all))`。

**提示**：增量解码时，新 token 可以看所有历史位置；因果掩码只需处理新 token 之间。


In [ ]:
class KVCacheAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cache=None):
        # x: (T_new, d); cache: (K_prev, V_prev) or None
        return None, cache   # TODO


torch.manual_seed(0)
kv10 = KVCacheAttention(8)
X10 = torch.randn(5, 8)

try:
    full_out10, _ = kv10(X10)
    outs10 = []
    cache10 = None
    for i in range(X10.size(0)):
        o10, cache10 = kv10(X10[i:i + 1], cache10)
        outs10.append(o10)
    step_out10 = torch.cat(outs10, dim=0)
except Exception:
    full_out10, step_out10, cache10 = None, None, None


In [ ]:
check("10.2a 全量输出形状为 (5,8)", lambda: tuple(full_out10.shape) == (5, 8))
check("10.2b 逐 token 解码与全量前向一致",
      lambda: close(step_out10, full_out10, 1e-5))
check("10.2c cache 累积到 5 个 token",
      lambda: cache10[0].shape == (5, 8))


### 练习 10.3　LoRA 线性层

**要做什么**

实现 `LoRALinear(base, r, alpha)`：

1. 冻结 `base` 的权重；
2. 创建 `A`（形状 `(r, in_features)`）与 `B`（形状 `(out_features, r)`），`B` 初始化为 0；
3. 前向：`base(x) + (alpha / r) * (x @ A.T @ B.T)`。

**提示**：可训练参数数应为 `r * (in + out)`；`B=0` 时输出应与 `base` 完全一致。


In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base = base
        self.r = r
        self.alpha = alpha
        # TODO: 冻结 base，创建 A、B

    def forward(self, x):
        # TODO
        return None


torch.manual_seed(0)
base10 = nn.Linear(8, 6)
lora10 = None
try:
    lora10 = LoRALinear(base10, r=4, alpha=8)
except Exception:
    pass
x10 = torch.randn(3, 8)
try:
    out_lora_init = lora10(x10)
except Exception:
    out_lora_init = None


In [ ]:
check("10.3a 构造出 A、B 两个低秩矩阵",
      lambda: lora10 is not None and hasattr(lora10, "A") and hasattr(lora10, "B"))
check("10.3b base 权重被冻结", lambda: lora10.base.weight.requires_grad is False)
check("10.3c 可训练参数数 = r*(in+out)",
      lambda: sum(p.numel() for p in lora10.parameters() if p.requires_grad) == 4 * (8 + 6))
check("10.3d B=0 时输出与 base 一致",
      lambda: close(out_lora_init, base10(x10), 1e-6))


In [ ]:
try:
    with torch.no_grad():
        lora10.B.normal_()
    out_lora_rand = lora10(x10)
    delta_ref = (8 / 4) * (x10 @ lora10.A.T @ lora10.B.T)
    lora_delta_ok = close(out_lora_rand - base10(x10), delta_ref, 1e-5)
except Exception:
    lora_delta_ok = False

check("10.3e 非零 B 时增量为 (alpha/r)*B*A*x", lambda: lora_delta_ok)


### 练习 10.4　SFT 的掩码损失

**要做什么**

实现 `masked_ce(logits, labels, ignore_index=-100)`：

- 把 `(B,T,V)` 展平成 `(B*T,V)`，标签展平成 `(B*T,)`；
- 只对标签不等于 `ignore_index` 的位置计算平均交叉熵。

**提示**：prompt 和 padding 的标签设成 $-100$，不会参与损失。


In [ ]:
def masked_ce(logits, labels, ignore_index=-100):
    return None   # TODO


torch.manual_seed(0)
logits10 = torch.randn(2, 4, 6)
labels10 = torch.tensor([
    [-100, -100, 2, 3],
    [-100, 1, 4, 5],
])

loss10 = None   # TODO


In [ ]:
_flat_logits = logits10.reshape(-1, 6)
_flat_labels = labels10.reshape(-1)
_mask10b = _flat_labels != -100
_ref_loss10 = F.cross_entropy(_flat_logits[_mask10b], _flat_labels[_mask10b])

check("10.4a 掩码损失与参考一致", lambda: close(loss10, _ref_loss10, 1e-6))


### 练习 10.5　DPO 损失

**要做什么**

实现 `dpo_loss(policy_logp_w, policy_logp_l, ref_logp_w, ref_logp_l, beta=0.1)`：

$$\mathcal{L}=-\log\sigma\left(\beta\left[(\log\pi_\theta(y_w)-\log\pi_{ref}(y_w))-(\log\pi_\theta(y_l)-\log\pi_{ref}(y_l))\right]\right)$$

**提示**：当策略与参考模型完全相同时，两个差值都为 0，损失应为 $\log 2$。


In [ ]:
def dpo_loss(policy_logp_w, policy_logp_l, ref_logp_w, ref_logp_l, beta=0.1):
    return None   # TODO


lp_w, lp_l = torch.tensor(-1.0), torch.tensor(-3.5)
rp_w, rp_l = torch.tensor(-2.0), torch.tensor(-2.5)
loss_dpo = None          # TODO: dpo_loss(lp_w, lp_l, rp_w, rp_l)
loss_dpo_equal = None    # TODO: 策略与参考相同时的损失


In [ ]:
_margin = 0.1 * ((torch.tensor(-1.0) - torch.tensor(-2.0))
                 - (torch.tensor(-3.5) - torch.tensor(-2.5)))
_ref_dpo = -torch.log(torch.sigmoid(_margin))

check("10.5a DPO 损失与公式一致",
      lambda: abs(loss_dpo.item() - _ref_dpo.item()) < 1e-6)
check("10.5b 策略=参考时损失为 log 2",
      lambda: abs(loss_dpo_equal.item() - math.log(2)) < 1e-6)


In [ ]:
scoreboard()


**第 10 章要点**：因果掩码保证只能看历史；KV cache 让增量解码与全量前向等价；LoRA 只训练低秩增量；SFT 只在回答上算损失；DPO 用参考模型约束偏好优化。


---

## 自测选择题

填 A/B/C/D，每题对应一章。

| 题号 | 题目 | 选项 |
| --- | --- | --- |
| 1 | `(4,1) + (1,3)` 的结果形状是 | A. (4,3)　B. (4,1)　C. (1,3)　D. 报错 |
| 2 | 线性回归的闭式解对应 | A. 梯度下降　B. 最小二乘　C. 早停　D. 数据增强 |
| 3 | 交叉熵对 logits 的梯度中，$p$ 是 | A. logits　B. softmax 概率　C. 标签　D. one-hot |
| 4 | 多层网络需要非线性激活，是因为 | A. 加速　B. 否则多层复合仍是线性　C. 省显存　D. 防过拟合 |
| 5 | LeNet-5 的参数主要集中在哪里 | A. 第一个卷积层　B. 池化层　C. 全连接层　D. 输入层 |
| 6 | LSTM 缓解梯度消失的关键是 | A. 加法细胞状态　B. 更多参数　C. 双向结构　D. 池化 |
| 7 | AdamW 相比 Adam 的主要改动是 | A. 解耦权重衰减　B. 去掉动量　C. 不需要学习率　D. 使用二阶差分 |
| 8 | 缩放点积注意力除以 $\sqrt{d_k}$ 是为了 | A. 防止 softmax 饱和　B. 加速　C. 正则化　D. 注入位置 |
| 9 | GIN 使用 sum 聚合的主要原因是 | A. 保留邻居多重性、提升表达力　B. 更快　C. 省显存　D. 防止过拟合 |
| 10 | LoRA 训练的是 | A. 全部参数　B. 低秩增量　C. 词表　D. 位置编码 |


In [ ]:
a1 = None; a2 = None; a3 = None; a4 = None; a5 = None
a6 = None; a7 = None; a8 = None; a9 = None; a10 = None


In [ ]:
quiz("1 广播", a1, "A")
quiz("2 闭式解", a2, "B")
quiz("3 CE 梯度", a3, "B")
quiz("4 非线性", a4, "B")
quiz("5 LeNet 参数", a5, "C")
quiz("6 LSTM", a6, "A")
quiz("7 AdamW", a7, "A")
quiz("8 缩放", a8, "A")
quiz("9 GIN", a9, "A")
quiz("10 LoRA", a10, "B")

scoreboard()


---

## 参考答案

先自己完成，再对照。每段代码都可以直接运行，但会重新定义对应练习中的变量；建议在新 kernel 中单独运行，或只复制你需要的部分。


### 第 1 章参考实现


In [ ]:
# ---- 1.1 ----
a = torch.arange(6., dtype=torch.float32).reshape(2, 3)
b = torch.tensor([10., 20., 30.])
out = a + b
shape_23_21 = tuple((a + torch.ones(2, 1)).shape)

err_type = None
try:
    a + torch.ones(2)
except RuntimeError as e:
    err_type = type(e).__name__

# ---- 1.2 ----
x1 = torch.tensor([1., 2., 3.], requires_grad=True)
y1 = (x1 ** 3).sum()
y1.backward()
grad1 = x1.grad.clone()

x2 = torch.tensor([1., 2., 3.], requires_grad=True)
w = torch.tensor(2.0, requires_grad=True)
b2 = torch.tensor(1.0, requires_grad=True)
loss2 = ((w * x2 + b2) ** 2).sum()
loss2.backward()
grad_w = w.grad.clone()
grad_b = b2.grad.clone()

# ---- 1.3 ----
class MyLinear(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_f, in_f))
        self.bias = nn.Parameter(torch.zeros(out_f))

    def forward(self, x):
        return x @ self.weight.T + self.bias


linear = MyLinear(3, 4)
x_lin = torch.randn(5, 3)
out_lin = linear(x_lin)

# ---- 1.4 ----
class ToyDataset(Dataset):
    def __init__(self, n=10):
        self.x = torch.arange(n, dtype=torch.float32)
        self.y = 2 * self.x + 1

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.y[i]


ds = ToyDataset()
loader = DataLoader(ds, batch_size=4, shuffle=False)
batches = list(loader)


### 第 2 章参考实现


In [ ]:
# ---- 2.1 ----
X = torch.tensor([[1.], [2.], [3.], [4.]], dtype=torch.float32)
y = 3 * X + 2
A = torch.cat([X, torch.ones(X.size(0), 1)], dim=1)
w = torch.linalg.lstsq(A, y).solution
pred = A @ w

# ---- 2.2 ----
xs = torch.linspace(-2, 2, 9).unsqueeze(1)
ys = 1 - 2 * xs + 3 * xs ** 2
A2 = torch.cat([torch.ones_like(xs), xs, xs ** 2], dim=1)
coef = torch.linalg.lstsq(A2, ys).solution
pred2 = A2 @ coef

# ---- 2.3 ----
class RunnerV1:
    def __init__(self, model):
        self.model = model

    def fit(self, X, y):
        design = torch.cat([X, torch.ones(X.size(0), 1)], dim=1)
        sol = torch.linalg.lstsq(design, y).solution
        with torch.no_grad():
            self.model.weight.copy_(sol[0:1].T)
            self.model.bias.copy_(sol[1].reshape(-1))

    def predict(self, X):
        return self.model(X)

    def evaluate(self, X, y):
        return F.mse_loss(self.predict(X), y).item()

    def save(self, path):
        torch.save(self.model.state_dict(), path)

    def load(self, path):
        self.model.load_state_dict(torch.load(path))


X_v1 = torch.tensor([[1.], [2.], [3.], [4.]], dtype=torch.float32)
y_v1 = 3 * X_v1 + 2

runner_v1 = RunnerV1(nn.Linear(1, 1))
runner_v1.fit(X_v1, y_v1)
pred_v1 = runner_v1.predict(X_v1)
mse_v1 = runner_v1.evaluate(X_v1, y_v1)

buf = io.BytesIO()
runner_v1.save(buf)
buf.seek(0)
runner_v1_loaded = RunnerV1(nn.Linear(1, 1))
runner_v1_loaded.load(buf)
pred_loaded = runner_v1_loaded.predict(X_v1)


### 第 3 章参考实现


In [ ]:
# ---- 3.1 ----
def logistic_grads(X, y, w, b):
    z = X @ w + b
    p = torch.sigmoid(z)
    dz = (p - y) / X.size(0)
    return X.T @ dz, dz.sum()


X3 = torch.tensor([[1., 2.], [2., -1.], [-1., 1.], [0., 3.]])
y3 = torch.tensor([1., 0., 0., 1.])
w3 = torch.tensor([0.5, -0.3])
b3 = torch.tensor(0.1)
grad_w, grad_b = logistic_grads(X3, y3, w3, b3)

# ---- 3.2 ----
def softmax_ce(logits, target):
    m = logits.max(dim=-1, keepdim=True).values
    logsumexp = (logits - m).exp().sum(-1, keepdim=True).log() + m
    log_probs = logits - logsumexp
    loss = -log_probs.gather(1, target[:, None]).squeeze(1).mean()
    p = log_probs.exp()
    onehot = F.one_hot(target, logits.size(-1)).float()
    grad = (p - onehot) / logits.size(0)
    return loss, grad


logits3 = torch.tensor([[2.0, 0.5, -1.0], [0.0, 1.0, 2.0], [1.0, 1.0, 1.0]])
target3 = torch.tensor([0, 2, 1])
loss3, grad3 = softmax_ce(logits3, target3)

# ---- 3.3 ----
class RunnerV2:
    def __init__(self, model, optimizer, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.best_state = None
        self.best_dev = float("inf")

    def train_epoch(self, loader):
        self.model.train()
        total, n = 0.0, 0
        for x, y in loader:
            logits = self.model(x)
            loss = self.loss_fn(logits, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            total += loss.item() * len(y)
            n += len(y)
        return total / n

    def evaluate(self, loader):
        self.model.eval()
        total, n = 0.0, 0
        with torch.no_grad():
            for x, y in loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * len(y)
                n += len(y)
        return total / n

    def fit(self, train_loader, dev_loader, epochs=8):
        history = {"train_loss": [], "dev_loss": []}
        for _ in range(epochs):
            history["train_loss"].append(self.train_epoch(train_loader))
            dev = self.evaluate(dev_loader)
            history["dev_loss"].append(dev)
            if dev < self.best_dev:
                self.best_dev = dev
                self.best_state = copy.deepcopy(self.model.state_dict())
        return history


torch.manual_seed(0)
Xc = torch.randn(200, 2)
yc = (Xc[:, 0] * Xc[:, 1] > 0).long()

train_loader = DataLoader(
    torch.utils.data.TensorDataset(Xc[:150], yc[:150]), batch_size=32, shuffle=True
)
dev_loader = DataLoader(
    torch.utils.data.TensorDataset(Xc[150:], yc[150:]), batch_size=32
)

model_v2 = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
runner_v2 = RunnerV2(
    model_v2,
    torch.optim.Adam(model_v2.parameters(), lr=0.05),
    nn.CrossEntropyLoss(),
)
history_v2 = runner_v2.fit(train_loader, dev_loader, epochs=8)


### 第 4 章参考实现


In [ ]:
# ---- 4.1 ----
def relu_t(x):
    return torch.clamp_min(x, 0)


def gelu_t(x):
    return 0.5 * x * (1 + torch.erf(x / math.sqrt(2)))


def silu_t(x):
    return x * torch.sigmoid(x)


def deriv(fn, x):
    xg = x.detach().clone().requires_grad_(True)
    y = fn(xg)
    g, = torch.autograd.grad(y.sum(), xg)
    return g


t4 = torch.linspace(-4, 4, 101)

# ---- 4.2 ----
torch.manual_seed(0)
X4 = torch.randn(6, 2)
y4 = torch.randn(6, 1)
W1_4 = torch.randn(2, 3)
b1_4 = torch.zeros(3)
W2_4 = torch.randn(3, 1)
b2_4 = torch.zeros(1)

z1 = X4 @ W1_4 + b1_4
a1 = torch.relu(z1)
z2 = a1 @ W2_4 + b2_4
loss4 = F.mse_loss(z2, y4)

dz2 = 2 * (z2 - y4) / y4.size(0)
dW2 = a1.T @ dz2
db2 = dz2.sum(0)
da1 = dz2 @ W2_4.T
dz1 = da1 * (z1 > 0).float()
dW1 = X4.T @ dz1
db1 = dz1.sum(0)

# ---- 4.3 ----
def accuracy(logits, y):
    return (logits.argmax(-1) == y).float().mean().item()


def is_better(cur, best, higher_is_better):
    return cur > best if higher_is_better else cur < best


def train_epoch_v3(model, loader, optimizer, loss_fn):
    model.train()
    total, n = 0.0, 0
    for x, y in loader:
        logits = model(x)
        loss = loss_fn(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
        n += len(y)
    return total / n


def evaluate_v3(model, loader, loss_fn, metric_fn):
    model.eval()
    total, n = 0.0, 0
    all_logits, all_y = [], []
    with torch.no_grad():
        for x, y in loader:
            logits = model(x)
            total += loss_fn(logits, y).item() * len(y)
            n += len(y)
            all_logits.append(logits)
            all_y.append(y)
    return total / n, metric_fn(torch.cat(all_logits), torch.cat(all_y))


torch.manual_seed(0)
X4b = torch.randn(200, 2)
y4b = (X4b[:, 0] * X4b[:, 1] > 0).long()
loader4 = DataLoader(torch.utils.data.TensorDataset(X4b, y4b), batch_size=32, shuffle=True)
model4 = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
opt4 = torch.optim.Adam(model4.parameters(), lr=0.05)
loss4b = nn.CrossEntropyLoss()

first_loss = train_epoch_v3(model4, loader4, opt4, loss4b)
for _ in range(2):
    train_epoch_v3(model4, loader4, opt4, loss4b)
dev_loss4, dev_acc4 = evaluate_v3(model4, loader4, loss4b, accuracy)


### 第 5 章参考实现


In [ ]:
# ---- 5.1 ----
def conv2d_naive(x, weight, bias=None, stride=1, padding=0):
    N, C, H, W = x.shape
    O, _, k, _ = weight.shape
    cols = F.unfold(x, kernel_size=k, padding=padding, stride=stride)
    out = weight.reshape(O, -1) @ cols
    H_out = (H + 2 * padding - k) // stride + 1
    W_out = (W + 2 * padding - k) // stride + 1
    out = out.view(N, O, H_out, W_out)
    if bias is not None:
        out = out + bias.view(1, O, 1, 1)
    return out


torch.manual_seed(0)
x5 = torch.randn(2, 3, 8, 8)
w5 = torch.randn(4, 3, 3, 3)
b5 = torch.randn(4)

out_same = conv2d_naive(x5, w5, b5, stride=1, padding=1)
out_down = conv2d_naive(x5, w5, b5, stride=2, padding=0)

# ---- 5.2 ----
lenet_shapes = {
    "C1": ((6, 28, 28), 156),
    "S2": ((6, 14, 14), 0),
    "C3": ((16, 10, 10), 2416),
    "S4": ((16, 5, 5), 0),
    "C5": ((120, 1, 1), 48120),
    "F6": ((84,), 10164),
    "F7": ((10,), 850),
}
lenet_total = 156 + 2416 + 48120 + 10164 + 850

# ---- 5.3 ----
def grad_norm_block(depth, use_residual, seed=0):
    torch.manual_seed(seed)
    layers = [nn.Linear(64, 64) for _ in range(depth)]
    for lin in layers:
        nn.init.xavier_uniform_(lin.weight)
    x = torch.randn(32, 64, requires_grad=True)
    h = x
    for lin in layers:
        z = F.relu(lin(h))
        h = h + z if use_residual else z
    h.pow(2).mean().backward()
    return x.grad.norm().item()


g_nores_20 = grad_norm_block(20, False)
g_res_20 = grad_norm_block(20, True)


### 第 6 章参考实现


In [ ]:
# ---- 6.1 ----
class MySRN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_xh = nn.Parameter(torch.randn(hidden_size, input_size))
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.b_ih = nn.Parameter(torch.zeros(hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x):
        B, T, D = x.shape
        H = self.W_hh.size(0)
        h = torch.zeros(B, H, device=x.device)
        outs = []
        for t in range(T):
            h = torch.tanh(x[:, t] @ self.W_xh.T + self.b_ih + h @ self.W_hh.T + self.b_hh)
            outs.append(h)
        return torch.stack(outs, dim=1), h.unsqueeze(0)


torch.manual_seed(0)
rnn6 = nn.RNN(3, 5, batch_first=True)
srn6 = MySRN(3, 5)
with torch.no_grad():
    srn6.W_xh.copy_(rnn6.weight_ih_l0)
    srn6.W_hh.copy_(rnn6.weight_hh_l0)
    srn6.b_ih.copy_(rnn6.bias_ih_l0)
    srn6.b_hh.copy_(rnn6.bias_hh_l0)

x6 = torch.randn(2, 4, 3)
out_srn, h_srn = srn6(x6)

# ---- 6.2 ----
torch.manual_seed(0)
lstm6 = nn.LSTM(4, 5)
x6b = torch.randn(1, 1, 4)
lstm_out6, _ = lstm6(x6b)

h0 = torch.zeros(1, 5)
c0 = torch.zeros(1, 5)
gates = (
    x6b[0] @ lstm6.weight_ih_l0.T
    + h0 @ lstm6.weight_hh_l0.T
    + lstm6.bias_ih_l0
    + lstm6.bias_hh_l0
)
i, f, g, o = gates.chunk(4, dim=1)
c = torch.sigmoid(f) * c0 + torch.sigmoid(i) * torch.tanh(g)
h_manual6 = torch.sigmoid(o) * torch.tanh(c)

# ---- 6.3 ----
def clip_grads(params, max_norm):
    total = torch.sqrt(sum((p.grad ** 2).sum() for p in params))
    if total > max_norm:
        scale = max_norm / (total + 1e-6)
        for p in params:
            p.grad.mul_(scale)
    return total.item()


torch.manual_seed(0)
params6 = [torch.randn(4, requires_grad=True) for _ in range(3)]
for p in params6:
    p.grad = torch.randn(4)

grad_before = torch.cat([p.grad.flatten() for p in params6]).clone()
norm_before = clip_grads(params6, 1.0)
grad_after = torch.cat([p.grad.flatten() for p in params6])


### 第 7 章参考实现


In [ ]:
# ---- 7.1 ----
def adam_step(p, grad, m, v, t, lr=0.01, b1=0.9, b2=0.999, eps=1e-8):
    m_new = b1 * m + (1 - b1) * grad
    v_new = b2 * v + (1 - b2) * grad ** 2
    m_hat = m_new / (1 - b1 ** t)
    v_hat = v_new / (1 - b2 ** t)
    p_new = p - lr * m_hat / (v_hat.sqrt() + eps)
    return p_new, m_new, v_new


torch.manual_seed(0)
p0 = torch.randn(3)
g0 = torch.randn(3)
m0 = torch.zeros(3)
v0 = torch.zeros(3)
p_new, m_new, v_new = adam_step(p0.clone(), g0, m0, v0, t=1, lr=0.01)

# ---- 7.2 ----
def std_ratio(init_fn, depth=30, width=256):
    torch.manual_seed(0)
    h = torch.randn(64, width)
    stds = []
    for _ in range(depth):
        lin = nn.Linear(width, width)
        init_fn(lin)
        h = F.relu(lin(h))
        stds.append(h.std().item())
    return stds[-1] / stds[0]


he_init = lambda lin: nn.init.kaiming_normal_(lin.weight, nonlinearity="relu")
default_init = lambda lin: nn.init.kaiming_uniform_(lin.weight, a=math.sqrt(5))

ratio_he = std_ratio(he_init)
ratio_default = std_ratio(default_init)

# ---- 7.3 ----
def bn_forward(x, gamma, beta, running_mean, running_var,
               momentum=0.1, eps=1e-5, training=True):
    if training:
        mu = x.mean(0)
        var = x.var(0, unbiased=False)
        var_unbiased = x.var(0, unbiased=True)
        new_rm = (1 - momentum) * running_mean + momentum * mu
        new_rv = (1 - momentum) * running_var + momentum * var_unbiased
        y = (x - mu) / torch.sqrt(var + eps) * gamma + beta
        return y, new_rm, new_rv
    y = (x - running_mean) / torch.sqrt(running_var + eps) * gamma + beta
    return y, running_mean, running_var


torch.manual_seed(0)
bn_ref = nn.BatchNorm1d(4)
x_bn = torch.randn(8, 4) * 2 + 5
gamma, beta = bn_ref.weight.detach().clone(), bn_ref.bias.detach().clone()

bn_ref.train()
y_train_ref = bn_ref(x_bn)
rm_ref = bn_ref.running_mean.clone()
rv_ref = bn_ref.running_var.clone()

y_train_mine, rm_mine, rv_mine = bn_forward(
    x_bn, gamma, beta, torch.zeros(4), torch.ones(4), training=True
)

bn_ref.eval()
y_eval_ref = bn_ref(x_bn)
y_eval_mine, _, _ = bn_forward(x_bn, gamma, beta, rm_ref, rv_ref, training=False)


### 第 8 章参考实现


In [ ]:
# ---- 8.1 ----
def additive_attention(q, K, V, Wq, Wk, v):
    scores = torch.tanh(Wq @ q + K @ Wk.T) @ v
    return scores.softmax(-1) @ V


def scaled_dot_attention(q, K, V):
    scores = (K @ q) / math.sqrt(q.size(0))
    return scores.softmax(-1) @ V


torch.manual_seed(0)
q8 = torch.randn(4)
K8 = torch.randn(5, 4)
V8 = torch.randn(5, 2)
Wq8 = torch.randn(3, 4)
Wk8 = torch.randn(3, 4)
v8 = torch.randn(3)

ctx_add = additive_attention(q8, K8, V8, Wq8, Wk8, v8)
ctx_dot = scaled_dot_attention(q8, K8, V8)

# ---- 8.2 ----
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, causal=False):
        B, T, d = x.shape
        H = self.n_heads
        dh = d // H
        q = self.q_proj(x).view(B, T, H, dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, H, dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, dh).transpose(1, 2)

        scores = q @ k.transpose(-2, -1) / math.sqrt(dh)
        if causal:
            mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
            scores = scores.masked_fill(mask, float("-inf"))
        attn = scores.softmax(-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, d)
        return self.out_proj(out)


torch.manual_seed(0)
mha8 = MultiHeadAttention(12, 3)
x8 = torch.randn(2, 5, 12)
out8 = mha8(x8, causal=True)
x8b = x8.clone()
x8b[:, -1] += 1.0
out8b = mha8(x8b, causal=True)

# ---- 8.3 ----
def sinusoidal_pe(seq_len, d_model):
    pos = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)
    i = torch.arange(0, d_model, 2, dtype=torch.float32)
    theta = 10000 ** (-i / d_model)
    ang = pos * theta
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(ang)
    pe[:, 1::2] = torch.cos(ang)
    return pe


pe8 = sinusoidal_pe(6, 8)


### 第 9 章参考实现


In [ ]:
# ---- 9.1 ----
def gcn_layer(A, H, W):
    A_tilde = A + torch.eye(A.size(0))
    D = A_tilde.sum(-1)
    D_inv_sqrt = torch.diag(D.pow(-0.5))
    A_norm = D_inv_sqrt @ A_tilde @ D_inv_sqrt
    out = F.relu(A_norm @ H @ W)
    return out, A_norm


torch.manual_seed(0)
N9 = 6
A9 = torch.zeros(N9, N9)
for i in range(N9):
    A9[i, (i + 1) % N9] = 1.0
    A9[(i + 1) % N9, i] = 1.0
H9 = torch.randn(N9, 4)
W9 = torch.randn(4, 5)

gcn_out9, gcn_A_norm9 = gcn_layer(A9, H9, W9)

# ---- 9.2 ----
def gat_layer(H, A, W, a, negative_slope=0.2):
    Wh = H @ W
    Fdim = W.size(1)
    term_i = Wh @ a[:Fdim].unsqueeze(1)
    term_j = Wh @ a[Fdim:].unsqueeze(1)
    e = F.leaky_relu(term_i + term_j.T, negative_slope)
    e = e.masked_fill(~A, float("-inf"))
    alpha = e.softmax(-1)
    return F.relu(alpha @ Wh), alpha


torch.manual_seed(0)
A9b = torch.tensor([
    [1, 1, 0, 0],
    [1, 1, 1, 0],
    [0, 1, 1, 1],
    [0, 0, 1, 1],
], dtype=torch.bool)
H9b = torch.randn(4, 3)
W9b = torch.randn(3, 2)
a9b = torch.randn(4)

gat_out9, gat_alpha9 = gat_layer(H9b, A9b, W9b, a9b)

# ---- 9.3 ----
def mean_cosine(X):
    Xn = F.normalize(X, dim=-1)
    S = Xn @ Xn.T
    n = S.size(0)
    off_diag = ~torch.eye(n, dtype=torch.bool)
    return S[off_diag].mean().item()


def smooth_sim(A_norm, X, layers):
    H = X
    for _ in range(layers):
        H = A_norm @ H
    return mean_cosine(H)


N9c = 10
A9c = torch.zeros(N9c, N9c)
for i in range(N9c):
    A9c[i, i] = 1.0
    A9c[i, (i + 1) % N9c] = 1.0
    A9c[i, (i - 1) % N9c] = 1.0
D9c = A9c.sum(-1)
A_norm9c = torch.diag(D9c.pow(-0.5)) @ A9c @ torch.diag(D9c.pow(-0.5))
torch.manual_seed(0)
X9c = torch.randn(N9c, 8)

sim2 = smooth_sim(A_norm9c, X9c, 2)
sim8 = smooth_sim(A_norm9c, X9c, 8)


### 第 10 章参考实现


In [ ]:
# ---- 10.1 ----
def causal_attention(q, k, v):
    T, d = q.shape
    scores = q @ k.T / math.sqrt(d)
    mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
    scores = scores.masked_fill(mask, float("-inf"))
    w = scores.softmax(-1)
    return w @ v, w


torch.manual_seed(0)
T10 = 6
q10 = torch.randn(T10, 8)
k10 = torch.randn(T10, 8)
v10 = torch.randn(T10, 8)
attn_out10, attn_w10 = causal_attention(q10, k10, v10)

# ---- 10.2 ----
class KVCacheAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cache=None):
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        if cache is None:
            K_all, V_all = k, v
        else:
            K_prev, V_prev = cache
            K_all = torch.cat([K_prev, k], dim=0)
            V_all = torch.cat([V_prev, v], dim=0)

        T_new = x.size(0)
        T_all = K_all.size(0)
        scores = q @ K_all.T / math.sqrt(x.size(1))
        mask = torch.triu(
            torch.ones(T_new, T_all, dtype=torch.bool, device=x.device),
            diagonal=T_all - T_new + 1,
        )
        scores = scores.masked_fill(mask, float("-inf"))
        out = scores.softmax(-1) @ V_all
        return out, (K_all, V_all)


torch.manual_seed(0)
kv10 = KVCacheAttention(8)
X10 = torch.randn(5, 8)
full_out10, _ = kv10(X10)
outs10 = []
cache10 = None
for i in range(X10.size(0)):
    o10, cache10 = kv10(X10[i:i + 1], cache10)
    outs10.append(o10)
step_out10 = torch.cat(outs10, dim=0)

# ---- 10.3 ----
class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base = base
        self.r = r
        self.alpha = alpha
        for p in base.parameters():
            p.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + (self.alpha / self.r) * (x @ self.A.T @ self.B.T)


torch.manual_seed(0)
base10 = nn.Linear(8, 6)
lora10 = LoRALinear(base10, r=4, alpha=8)
x10 = torch.randn(3, 8)
out_lora_init = lora10(x10)

with torch.no_grad():
    lora10.B.normal_()
out_lora_rand = lora10(x10)

# ---- 10.4 ----
def masked_ce(logits, labels, ignore_index=-100):
    flat_logits = logits.reshape(-1, logits.size(-1))
    flat_labels = labels.reshape(-1)
    mask = flat_labels != ignore_index
    return F.cross_entropy(flat_logits[mask], flat_labels[mask])


torch.manual_seed(0)
logits10 = torch.randn(2, 4, 6)
labels10 = torch.tensor([
    [-100, -100, 2, 3],
    [-100, 1, 4, 5],
])
loss10 = masked_ce(logits10, labels10)

# ---- 10.5 ----
def dpo_loss(policy_logp_w, policy_logp_l, ref_logp_w, ref_logp_l, beta=0.1):
    margin = beta * ((policy_logp_w - ref_logp_w) - (policy_logp_l - ref_logp_l))
    return -torch.log(torch.sigmoid(margin))


lp_w, lp_l = torch.tensor(-1.0), torch.tensor(-3.5)
rp_w, rp_l = torch.tensor(-2.0), torch.tensor(-2.5)
loss_dpo = dpo_loss(lp_w, lp_l, rp_w, rp_l)
loss_dpo_equal = dpo_loss(lp_w, lp_l, lp_w, lp_l)

# ---- quiz ----
a1 = "A"; a2 = "B"; a3 = "B"; a4 = "B"; a5 = "C"
a6 = "A"; a7 = "A"; a8 = "A"; a9 = "A"; a10 = "B"
